# Лабораторная работа 5. Диффузионные модели (Stable Diffusion)

В этой лабораторной работе мы дообучим модель **Stable Diffusion v1.5** с помощью метода **DreamBooth** и **LoRA**, чтобы она научилась генерировать изображения с конкретным человеком (мной).

**Цели:**
1. Настроить окружение и загрузить базовую модель.
2. Сгенерировать изображения класса (Prior Preservation) для предотвращения деградации модели.
3. Обучить LoRA-адаптер на 10-15 фотографиях пользователя.
4. Сгенерировать изображения в разных стилях (киберпанк, ведьмак и т.д.).
5. Оценить качество генераций с помощью метрик (Face Similarity, CLIP Score, BRISQUE).


## §1. Dev — Подготовка окружения
Клонирование или обновление репозитория (Git Pull).


In [ ]:
import os
import sys
from pathlib import Path

if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ:
    from google.colab import userdata  # type: ignore
    github_token = userdata.get('GITHUB_TOKEN')
    repo_url = f"https://{github_token}@github.com/Ma-XD/ITMO-CV.git"
    
    if not Path("/content/ITMO-CV").exists():
        !git clone {repo_url} /content/ITMO-CV
    else:
        %cd /content/ITMO-CV
        !git pull
    
    sys.path.insert(0, "/content/ITMO-CV/lab5-DIF")
    %cd /content/ITMO-CV/lab5-DIF
else:
    LAB_DIR = Path(".").resolve()
    if LAB_DIR.name != "lab5-DIF":
        %cd lab5-DIF
    if str(LAB_DIR) not in sys.path:
        sys.path.insert(0, str(LAB_DIR))


## §2. Установка зависимостей
Монтирование Google Drive (в Colab) и установка библиотек.


In [ ]:
from env_config import is_colab

if is_colab:
    from google.colab import drive  # type: ignore
    # drive.mount идемпотентен: если уже примонтирован, он просто напишет 'Drive already mounted'
    drive.mount('/content/drive')
    
    # pip install идемпотентен: если пакеты уже установлены, он напишет 'Requirement already satisfied'
    !pip install -r requirements.txt


## §3. Импорты и проверка конфигурации
Загрузка путей из `config.py` и проверка доступности GPU.


In [ ]:
import sys
from pathlib import Path
import torch
import matplotlib.pyplot as plt
from PIL import Image

# Добавляем папку лабы в sys.path для импорта конфигов
lab_dir = Path("lab5-DIF").resolve() if not is_colab else Path("/content/ITMO-CV/lab5-DIF")
if str(lab_dir) not in sys.path:
    sys.path.insert(0, str(lab_dir))

from env_config import print_env, get_device
from config import (
    INSTANCE_DIR, CLASS_DIR, CHECKPOINT_DIR, FIGURE_DIR,
    MODEL_ID, INSTANCE_PROMPT, CLASS_PROMPT, RESOLUTION
)

print_env()
device = get_device()

print(f"\nПапка с фото пользователя (Instance): {INSTANCE_DIR}")
print(f"Папка с фото класса (Class): {CLASS_DIR}")
print(f"Чекпоинты сохраняются в: {CHECKPOINT_DIR}")


## §4. Загрузка базовой модели
Загружаем оригинальную модель Stable Diffusion v1.5 (идемпотентно).


In [ ]:
from diffusers import StableDiffusionPipeline
import torch

try:
    if pipeline is not None:
        print("✅ Базовая модель уже находится в памяти.")
except NameError:
    print("Загрузка базовой модели SD 1.5...")
    pipeline = StableDiffusionPipeline.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16 if device.type == "cuda" else torch.float32,
        safety_checker=None  # Отключаем safety checker
    )
    pipeline = pipeline.to(device)
    if device.type == "cuda":
        pipeline.enable_attention_slicing()
    print("✅ Базовая модель успешно загружена!")


### Smoke-test базовой модели
Генерируем тестовое изображение до дообучения, чтобы убедиться в работоспособности.


In [ ]:
# Smoke-test: генерация тестового изображения
prompt = "a photo of a cat"
generator = torch.Generator(device=device).manual_seed(42)

print(f"Генерируем: '{prompt}'...")
test_image = pipeline(prompt, generator=generator, num_inference_steps=30).images[0]

# Отрисовка результата
plt.figure(figsize=(6, 6))
plt.imshow(test_image)
plt.axis('off')
plt.title("Smoke-test: Базовая модель SD 1.5")
plt.show()


## §5. Генерация изображений класса (Prior Preservation)
Создаем 150 изображений обычных людей, чтобы модель не забыла их после дообучения на лице пользователя. Ячейка идемпотентна.


In [ ]:
import hashlib
from tqdm.auto import tqdm

num_class_images = 150
existing_images = list(CLASS_DIR.glob("*.jpg")) + list(CLASS_DIR.glob("*.png"))

if len(existing_images) >= num_class_images:
    print(f"✅ Изображения класса уже существуют ({len(existing_images)} шт.). Пропускаем генерацию.")
else:
    images_to_generate = num_class_images - len(existing_images)
    print(f"Сбор изображений класса. Нужно сгенерировать {images_to_generate} шт.")
    
    # Отключаем градиенты для ускорения генерации
    with torch.no_grad():
        for i in tqdm(range(images_to_generate), desc="Генерация class images"):
            # Генерируем изображение
            image = pipeline(CLASS_PROMPT, num_inference_steps=30).images[0]
            
            # Сохраняем с уникальным именем (хэш)
            hash_name = hashlib.sha1(image.tobytes()).hexdigest()[:10]
            image_path = CLASS_DIR / f"class_{hash_name}.jpg"
            image.save(image_path)
            
    print("✅ Генерация изображений класса завершена.")


## §6. Обучение DreamBooth LoRA
Напишем класс датасета и сам цикл обучения. Ячейка обучения идемпотентна.

### Автоматический ресайз фотографий пользователя
Если вы загрузили квадратные фото с телефона (например, 1024x1024 или 2048x2048), эта ячейка автоматически сожмет их до нужных 512x512, чтобы сэкономить память и ускорить загрузку.

In [ ]:
from PIL import Image

def resize_instance_images(instance_dir, size=512):
    paths = [p for p in Path(instance_dir).iterdir() if p.is_file() and p.suffix.lower() in ['.jpg', '.jpeg', '.png']]
    if not paths:
        print(f"⚠️ Папка {instance_dir} пуста. Загрузите свои фотографии!")
        return
        
    resized_count = 0
    for p in paths:
        with Image.open(p) as img:
            if img.size != (size, size):
                # Используем LANCZOS для качественного сжатия
                img_resized = img.resize((size, size), Image.Resampling.LANCZOS)
                img_resized.save(p)
                resized_count += 1
                
    if resized_count > 0:
        print(f"✅ Успешно изменено разрешение {resized_count} фотографий до {size}x{size}.")
    else:
        print(f"✅ Все фотографии уже имеют размер {size}x{size}.")

resize_instance_images(INSTANCE_DIR, RESOLUTION)

In [ ]:
import random
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class DreamBoothDataset(Dataset):
    def __init__(self, instance_dir, class_dir, tokenizer, size=512):
        self.instance_paths = [p for p in Path(instance_dir).iterdir() if p.is_file() and p.suffix.lower() in ['.jpg', '.jpeg', '.png']]
        self.class_paths = [p for p in Path(class_dir).iterdir() if p.is_file() and p.suffix.lower() in ['.jpg', '.jpeg', '.png']]
        
        if not self.instance_paths:
            raise ValueError(f"❌ Папка {instance_dir} пуста! Добавьте свои фото.")
            
        self.tokenizer = tokenizer
        self.size = size
        self.transform = transforms.Compose([
            transforms.Resize((size, size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5])
        ])

    def __len__(self):
        # Искусственно увеличиваем длину эпохи
        return len(self.instance_paths) * 100

    def __getitem__(self, index):
        inst_path = self.instance_paths[index % len(self.instance_paths)]
        class_path = random.choice(self.class_paths)

        inst_img = Image.open(inst_path).convert("RGB")
        class_img = Image.open(class_path).convert("RGB")

        inst_prompt_ids = self.tokenizer(
            INSTANCE_PROMPT, truncation=True, padding="max_length", 
            max_length=self.tokenizer.model_max_length, return_tensors="pt"
        ).input_ids[0]

        class_prompt_ids = self.tokenizer(
            CLASS_PROMPT, truncation=True, padding="max_length", 
            max_length=self.tokenizer.model_max_length, return_tensors="pt"
        ).input_ids[0]

        return {
            "instance_images": self.transform(inst_img),
            "instance_prompt_ids": inst_prompt_ids,
            "class_images": self.transform(class_img),
            "class_prompt_ids": class_prompt_ids
        }

In [ ]:
import torch.nn.functional as F
from peft import LoraConfig, get_peft_model
from diffusers import DDPMScheduler
from tqdm.auto import tqdm

lora_path = CHECKPOINT_DIR / "pytorch_lora_weights.safetensors"

if lora_path.exists():
    print("✅ LoRA уже обучена и сохранена. Пропускаем обучение.")
else:
    print("Подготовка к обучению...")
    unet = pipeline.unet
    text_encoder = pipeline.text_encoder
    vae = pipeline.vae
    tokenizer = pipeline.tokenizer
    noise_scheduler = DDPMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")

    # Замораживаем базовые веса
    vae.requires_grad_(False)
    text_encoder.requires_grad_(False)
    unet.requires_grad_(False)

    # Настраиваем LoRA (только для UNet attention)
    lora_config = LoraConfig(
        r=8, lora_alpha=8, 
        target_modules=["to_q", "to_k", "to_v", "to_out.0"]
    )
    unet = get_peft_model(unet, lora_config)
    unet.to(device)

    # Датасет и оптимизатор
    dataset = DreamBoothDataset(INSTANCE_DIR, CLASS_DIR, tokenizer, RESOLUTION)
    dataloader = DataLoader(dataset, batch_size=1, shuffle=True)
    optimizer = torch.optim.AdamW(unet.parameters(), lr=1e-4)

    num_steps = 1000
    progress_bar = tqdm(total=num_steps, desc="Обучение LoRA")
    step = 0

    unet.train()
    while step < num_steps:
        for batch in dataloader:
            if step >= num_steps:
                break

            # Конкатенируем instance и class для Prior Preservation
            images = torch.cat([batch["instance_images"], batch["class_images"]]).to(device)
            prompt_ids = torch.cat([batch["instance_prompt_ids"], batch["class_prompt_ids"]]).to(device)

            # Переводим картинки в латентное пространство
            latents = vae.encode(images.to(dtype=vae.dtype)).latent_dist.sample()
            latents = latents * vae.config.scaling_factor

            # Добавляем шум
            noise = torch.randn_like(latents)
            bsz = latents.shape[0]
            timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (bsz,), device=device).long()
            noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

            # Получаем эмбеддинги текста
            encoder_hidden_states = text_encoder(prompt_ids)[0]

            # Предсказываем шум
            model_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample

            # Считаем Loss (Prior Preservation: вес 1.0)
            model_pred_inst, model_pred_class = model_pred.chunk(2)
            noise_inst, noise_class = noise.chunk(2)

            loss_inst = F.mse_loss(model_pred_inst, noise_inst, reduction="mean")
            loss_class = F.mse_loss(model_pred_class, noise_class, reduction="mean")
            loss = loss_inst + loss_class

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            progress_bar.update(1)
            progress_bar.set_postfix({"loss": loss.item()})
            step += 1

    # Сохраняем веса в формате diffusers
    from peft.utils import get_peft_model_state_dict
    from diffusers import StableDiffusionPipeline
    
    unet_lora_state_dict = get_peft_model_state_dict(unet)
    StableDiffusionPipeline.save_lora_weights(
        save_directory=str(CHECKPOINT_DIR),
        unet_lora_layers=unet_lora_state_dict,
        safe_serialization=True,
    )
    print(f"✅ Обучение завершено! Веса LoRA сохранены в {CHECKPOINT_DIR}")

## §7. Инференс и генерация
Загружаем обученную LoRA поверх базовой модели и генерируем изображения по заданию.

In [ ]:
import gc
import torch
from diffusers import StableDiffusionPipeline

print("Очистка памяти и перезагрузка чистой базовой модели...")
# Удаляем старый pipeline с PEFT-оберткой
if 'pipeline' in globals():
    del pipeline
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()

# Загружаем чистую базовую модель
pipeline = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if device.type == "cuda" else torch.float32,
    safety_checker=None
).to(device)

if device.type == "cuda":
    pipeline.enable_attention_slicing()

# Загружаем веса LoRA в чистый пайплайн
print("Загрузка весов LoRA...")
pipeline.load_lora_weights(CHECKPOINT_DIR)
print("✅ LoRA успешно применена к чистой модели!")

### 7.1. Генерация в разных стилях
Генерируем 5 качественных изображений пользователя в разном окружении.

In [ ]:
# Промпты для 5 разных стилей
style_prompts = [
    f"{INSTANCE_PROMPT} in a neon-lit cyberpunk city street, wearing futuristic techwear, highly detailed, Blade Runner style",
    f"{INSTANCE_PROMPT} as a dwarf warrior in a massive underground dwarven kingdom, surrounded by gold and glowing forges, fantasy art",
    f"{INSTANCE_PROMPT} wearing a Vault suit in a post-apocalyptic wasteland, ruined buildings, retro-futuristic, Fallout style",
    f"{INSTANCE_PROMPT} as a Space Marine in heavy power armor, epic battle, grimdark universe, Warhammer 40k style",
    f"{INSTANCE_PROMPT} as a Witcher, wearing leather armor, carrying two swords on back, dark fantasy forest, The Witcher 3 style"
]

titles = ["Cyberpunk", "Dwarven Kingdom", "Fallout", "Warhammer 40k", "The Witcher"]

generator = torch.Generator(device=device).manual_seed(42)
style_images = []

print("Генерация изображений в разных стилях...")
for prompt in style_prompts:
    img = pipeline(prompt, generator=generator, num_inference_steps=30).images[0]
    style_images.append(img)

# Отрисовка
fig, axes = plt.subplots(1, 5, figsize=(20, 5))
for ax, img, title in zip(axes, style_images, titles):
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(title)
plt.tight_layout()
plt.show()

### 7.2. Генерация в лесу, городе и на пляже
Проверка работы токена в базовых локациях с уточнением `high quality, realism`.

In [ ]:
location_prompts = [
    f"{INSTANCE_PROMPT} in a forest, high quality, realism",
    f"{INSTANCE_PROMPT} in a city, high quality, realism",
    f"{INSTANCE_PROMPT} in a beach, high quality, realism"
]

loc_titles = ["Forest", "City", "Beach"]
loc_images = []

print("Генерация в базовых локациях...")
for prompt in location_prompts:
    img = pipeline(prompt, generator=generator, num_inference_steps=30).images[0]
    loc_images.append(img)

# Отрисовка
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, img, title in zip(axes, loc_images, loc_titles):
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(title)
plt.tight_layout()
plt.show()

### 7.3. Контрольные промпты по заданию (проверка сохранности класса)
Генерируем изображения по тексту `"a photo of a man in a forest"`, `"... in a city"`, `"... in a beach"` с уточнениями `high quality, realism`.

**Важно:** LoRA активна (scale=1.0), не выгружена. Эти изображения **НЕ** должны быть похожи на пользователя — это доказывает, что модель не переобучилась и не разучилась генерировать обычных мужчин.

In [ ]:
control_prompts = [
    "a photo of a man in a forest, high quality, realism",
    "a photo of a man in a city, high quality, realism",
    "a photo of a man in a beach, high quality, realism",
]
control_titles = ["Man in forest", "Man in city", "Man in beach"]
control_images = []

print("Генерация контрольных промптов (без токена ohwx)...")
for prompt, seed in zip(control_prompts, [101, 202, 303]):
    gen = torch.Generator(device=device).manual_seed(seed)
    img = pipeline(prompt, generator=gen, num_inference_steps=30).images[0]
    control_images.append(img)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, img, title in zip(axes, control_images, control_titles):
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(title)
plt.tight_layout()
plt.show()

## §8. Оценка качества (Метрики)

Согласно заданию (п. 6), необходимо оценить качество сгенерированных изображений 3 мерами (не используя FID и IS) и обосновать их выбор.

### Обоснование выбора метрик:
1. **Face Similarity (через InceptionResnetV1 / FaceNet)**: 
   - *Зачем:* Главная цель DreamBooth — сохранить портретное сходство (Identity Preservation). Эта метрика извлекает эмбеддинги лиц из реальных фото и сгенерированных, а затем считает косинусное расстояние между ними. Чем ближе к 1, тем больше сгенерированный человек похож на оригинал.
2. **CLIP Score**:
   - *Зачем:* Оценивает Text-Image Alignment (насколько картинка соответствует текстовому промпту). Важно понять, не игнорирует ли модель фон (например, рисует ли она лес, если мы просили лес) из-за переобучения на лице.
3. **BRISQUE (через `piq`)**:
   - *Зачем:* Безэталонная метрика (No-Reference Image Quality Assessment). Оценивает общее визуальное качество картинки (наличие шумов, артефактов, искажений) на основе статистики естественных изображений. Меньше значение — лучше качество.

In [ ]:
import torch
import torchvision.transforms as T
from facenet_pytorch import InceptionResnetV1, MTCNN
import open_clip
import piq
import numpy as np

print("Загрузка моделей для метрик...")

# 1. Модели для Face Similarity
mtcnn = MTCNN(image_size=160, margin=0, device=device)
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

# 2. Модель для CLIP Score
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k', device=device)
clip_tokenizer = open_clip.get_tokenizer('ViT-B-32')

# 3. Трансформы для BRISQUE
brisque_transform = T.Compose([
    T.ToTensor(),
])

print("✅ Модели загружены!")

### Расчет метрик на сгенерированных изображениях

Чтобы доказать пункт 4 задания (модель не разучилась генерировать других людей), считаем Face Similarity отдельно на двух группах: с токеном `ohwx` и без него. Высокий разрыв между ними — главное численное доказательство корректности дообучения.

In [ ]:
import numpy as np

# 1. Извлекаем эталонный эмбеддинг (усредняем по всем фото пользователя)
ref_embs = []
for ref_path in INSTANCE_DIR.glob("*"):
    if ref_path.suffix.lower() not in ['.jpg', '.jpeg', '.png']:
        continue
    ref_img = Image.open(ref_path).convert("RGB")
    ref_face = mtcnn(ref_img)
    if ref_face is not None:
        ref_embs.append(resnet(ref_face.unsqueeze(0).to(device)))

if not ref_embs:
    raise RuntimeError("❌ Не удалось извлечь лица из эталонных фото")

ref_emb = torch.stack(ref_embs).mean(dim=0)
ref_emb = ref_emb / ref_emb.norm(dim=-1, keepdim=True)

# 2. Функция для расчета метрик на группе изображений
def compute_metrics(images, prompts, label):
    face_sims, clip_s, brisque_s = [], [], []
    with torch.no_grad():
        for img, prompt in zip(images, prompts):
            # Face Similarity
            face = mtcnn(img)
            if face is not None:
                emb = resnet(face.unsqueeze(0).to(device))
                emb = emb / emb.norm(dim=-1, keepdim=True)
                face_sims.append(torch.cosine_similarity(ref_emb, emb).item())
            
            # CLIP
            img_in = clip_preprocess(img).unsqueeze(0).to(device)
            txt_in = clip_tokenizer([prompt]).to(device)
            im_f = clip_model.encode_image(img_in)
            tx_f = clip_model.encode_text(txt_in)
            im_f = im_f / im_f.norm(dim=-1, keepdim=True)
            tx_f = tx_f / tx_f.norm(dim=-1, keepdim=True)
            clip_s.append((im_f @ tx_f.T).item())
            
            # BRISQUE
            try:
                t = brisque_transform(img).unsqueeze(0).to(device)
                v = piq.brisque(t, data_range=1.0).item()
                if not np.isnan(v):
                    brisque_s.append(v)
            except Exception as e:
                print(f"BRISQUE skip ({label}): {e}")
                
        return {
            "face_sim": np.mean(face_sims) if face_sims else 0.0,
            "clip": np.mean(clip_s),
            "brisque": np.mean(brisque_s) if brisque_s else 0.0,
            "n_faces_detected": len(face_sims),
        }

print("Расчет метрик для изображений с токеном...")
metrics_ohwx = compute_metrics(
    style_images + loc_images, 
    style_prompts + location_prompts, 
    "ohwx"
)

print("Расчет метрик для контрольных изображений...")
# У нас осталась только группа control_images (т.к. val_images мы удалили в 7.4)
metrics_control = compute_metrics(
    control_images, 
    control_prompts, 
    "control"
)

# 3. Вывод результатов
print("\n" + "=" * 70)
print(f"{'Метрика':<25}{'С токеном ohwx':<22}{'Без токена (control)':<22}")
print("=" * 70)
print(f"{'Face Similarity ↑':<25}{metrics_ohwx['face_sim']:<22.4f}{metrics_control['face_sim']:<22.4f}")
print(f"{'CLIP Score ↑':<25}{metrics_ohwx['clip']:<22.4f}{metrics_control['clip']:<22.4f}")
print(f"{'BRISQUE ↓':<25}{metrics_ohwx['brisque']:<22.4f}{metrics_control['brisque']:<22.4f}")
print("=" * 70)
print("Ожидаемая интерпретация:")
print("  - Face Sim высокая для ohwx и низкая для control → identity сохранён, класс не сломан")
print("  - CLIP Score сопоставим в обеих группах → модель понимает промпт одинаково")
print("  - BRISQUE сопоставим → визуальное качество не деградировало")